<a href="https://colab.research.google.com/github/srikanthkakumanu/do-hugging-face/blob/master/llm-engineering-hf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# LLM Engineering

We perform the following action items using Transformers framework.

 1. Sentiment Analysis with IMDB dataset
 2. Named Entity Recognition
 3. Question Answering
 4. Machine Translation

## Sentiment Analysis

In [6]:
from transformers import pipeline
from datasets import load_dataset
import pandas as pd

In [ ]:
ds = load_dataset('stanfordnlp/imdb')

In [7]:
type(ds)

datasets.dataset_dict.DatasetDict

In [8]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [9]:
ds["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})

In [10]:
ds["train"].to_pandas()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0
...,...,...
24995,A hit at the time but now better categorised a...,1
24996,I love this movie like no other. Another time ...,1
24997,This film and it's sequel Barry Mckenzie holds...,1
24998,'The Adventures Of Barry McKenzie' started lif...,1


In [11]:
my_dataset_df = ds["train"].to_pandas()

In [12]:
my_dataset_df["text"]

,text
0,I rented I AM CURIOUS-YELLOW from my video sto...
1,"""I Am Curious: Yellow"" is a risible and preten..."
2,If only to avoid making this type of film in t...
3,This film was probably inspired by Godard's Ma...
4,"Oh, brother...after hearing about this ridicul..."
...,...
24995,A hit at the time but now better categorised a...
24996,I love this movie like no other. Another time ...
24997,This film and it's sequel Barry Mckenzie holds...
24998,'The Adventures Of Barry McKenzie' started lif...


In [13]:
len(my_dataset_df["text"])

25000

In [ ]:
classifier = pipeline("sentiment-analysis")

In [15]:
classifier("This day is great!")

[{'label': 'POSITIVE', 'score': 0.999881386756897}]

In [16]:
classifier("This day is terrible and i am so sad")[0]["label"]

'NEGATIVE'

In [17]:
def score(review_text):
    return classifier(review_text[:500])[0]["label"]

In [ ]:
my_dataset_df["model_prediction"] = my_dataset_df["text"].apply(score)

In [ ]:
my_dataset_df[["label", "model_prediction"]][:20]

In [ ]:
my_dataset_df.iloc[0]

In [ ]:
review = my_dataset_df.iloc[0]["text"]
classifier(review)[0]["label"]

In [ ]:
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")

In [20]:
sentence = "The company reported a strong increase in quarterly revnue, exceeding market expectations."
finbert(sentence)

[{'label': 'positive', 'score': 0.951565146446228}]

In [21]:
sentence = "Shares fell after the firm reported lower-than-expected earnings"
finbert(sentence)

[{'label': 'negative', 'score': 0.9718543887138367}]

In [23]:
sentences = ["Strong consumer demand drove record sales across all regions",
             "Supply chain disruptions severly affected production output"]

In [24]:
finbert(sentences)

[{'label': 'positive', 'score': 0.9434744715690613},
 {'label': 'negative', 'score': 0.9608097672462463}]

## Named Entity Recognition

In [25]:

sentence = "Apple announced record earnings in the United States on Monday."

In [ ]:

ner = pipeline("ner")

In [27]:
sentence

'Apple announced record earnings in the United States on Monday.'

In [28]:
ner(sentence)

[{'entity': 'I-ORG',
  'score': np.float32(0.9991387),
  'index': 1,
  'word': 'Apple',
  'start': 0,
  'end': 5},
 {'entity': 'I-LOC',
  'score': np.float32(0.9997105),
  'index': 7,
  'word': 'United',
  'start': 39,
  'end': 45},
 {'entity': 'I-LOC',
  'score': np.float32(0.99967074),
  'index': 8,
  'word': 'States',
  'start': 46,
  'end': 52}]

In [29]:
sentence = "I live in UK worked at Facebook after graduating from Harvard"

In [30]:
ner(sentence)

[{'entity': 'I-LOC',
  'score': np.float32(0.9997962),
  'index': 4,
  'word': 'UK',
  'start': 10,
  'end': 12},
 {'entity': 'I-ORG',
  'score': np.float32(0.9983279),
  'index': 7,
  'word': 'Facebook',
  'start': 23,
  'end': 31},
 {'entity': 'I-ORG',
  'score': np.float32(0.9209188),
  'index': 11,
  'word': 'Harvard',
  'start': 54,
  'end': 61}]

## Question Answering

In [34]:
qa_bot = pipeline("question-answering")

No model was supplied, defaulted to distilbert/distilbert-base-cased-distilled-squad and revision 564e9b5.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [35]:
context = """
Financial sentiment analysis is a challenging task due to the specialized
language and lack of labeled data in that domain. General-purpose models are
not effective enough because of the specialized language used in a financial
context. We hypothesize that pre-trained language models can help with this
problem because they require fewer labeled examples and they can be further
trained on domain-specific corpora. We introduce FinBERT, a language model
based on BERT, to tackle NLP tasks in the financial domain. Our results show
improvement in every measured metric on current state-of-the-art results for
two financial sentiment analysis datasets. We find that even with a smaller
training set and fine-tuning only a part of the model, FinBERT outperforms
state-of-the-art machine learning methods.
"""

In [37]:
question = "What is financial sentiment analysis?"

In [38]:
qa_bot(question=question, context=context)

{'score': 0.302753210067749,
 'start': 33,
 'end': 51,
 'answer': 'a challenging task'}

In [39]:
question = "What is FinBERT?"

In [40]:
result = qa_bot(question=question, context=context)
print(result["answer"])

a language model
based on BERT


## Machine Translation

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

translater = pipeline("translation_en_to_fr")

In [ ]:
translater("Hello")

In [ ]:
translater("Thanks")

In [ ]:
sentence = "What is your name?"
translater(sentence)[0]["translation_text"]

In [48]:

# Use a pipeline as a high-level helper
from transformers import pipeline

In [ ]:
pipe = pipeline("translation", model="facebook/nllb-200-distilled-600M")